# 09 — Explainability + Fairness

**Owner:** Person B (decision track).

**Rubric line:** Explanation framework that's defensible to non-technical
stakeholders (compliance officers, ops leadership). Plus a fairness
sanity-check.


In [ ]:
# --- Setup --------------------------------------------------------------
# Make `src/` importable regardless of where the notebook is launched from.
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config, data, features, models, metrics, decision, viz

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 9.1 — Load split + improved model

In [ ]:
import joblib
train_df, test_df = data.load_interim()
feature_cols = data.feature_columns(include_leaky=False)
X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_test = test_df[config.TARGET_COL]
lgbm_artifact = joblib.load(config.MODELS_DIR / 'improved_lgbm.joblib')
calibrated = lgbm_artifact['model']
y_proba = lgbm_artifact['proba_test']


## 9.2 — SHAP global feature importance

Reach into the underlying LightGBM estimator (CalibratedClassifierCV
wraps it). SHAP runs on the unwrapped tree model.


In [ ]:
import shap
# Pull the underlying fitted LightGBM out of the calibration wrapper.
inner = calibrated.calibrated_classifiers_[0].estimator
preproc = inner.named_steps['preprocessor']
lgbm = inner.named_steps['classifier']
X_test_t = preproc.transform(X_test)
feature_names = preproc.get_feature_names_out()
explainer = shap.TreeExplainer(lgbm)
shap_values = explainer.shap_values(X_test_t)
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # positive class
shap.summary_plot(shap_values, X_test_t, feature_names=feature_names, show=False)
plt.gcf().savefig(config.FIGURES_DIR / '09_shap_summary.png', bbox_inches='tight', dpi=200)


## 9.3 — SHAP local explanations (one yes-target, one no-target)

Pick two real test customers — one the model strongly recommends contacting,
one it strongly recommends skipping — and show the per-feature contributions.
These are the slides that make the model defensible to a compliance officer.


## 9.4 — Decision-rule extraction (shallow tree)

Train a depth-3 decision tree to mimic the LightGBM scores. The resulting
rules are human-readable and useful as a fallback / training aid for
front-line staff.


In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text
surrogate = DecisionTreeRegressor(max_depth=3, random_state=config.RANDOM_SEED)
surrogate.fit(X_test_t, y_proba)
rules_text = export_text(surrogate, feature_names=list(feature_names))
print(rules_text)
(config.TABLES_DIR / 'surrogate_rules.txt').write_text(rules_text)


## 9.5 — Fairness sanity-check

Compare model performance and contact rates across age bands and education
levels. We're not enforcing a strict fairness criterion — we're surfacing
any disparities so a compliance officer can interrogate them.


In [ ]:
test_aug = features.add_engineered_features(test_df).assign(proba=y_proba)
from sklearn.metrics import average_precision_score

def group_perf(df, group_col):
    rows = []
    for name, g in df.groupby(group_col, observed=True):
        if g[config.TARGET_COL].nunique() < 2:
            continue
        rows.append({
            group_col: name,
            'n': len(g),
            'base_rate': g[config.TARGET_COL].mean(),
            'mean_predicted': g['proba'].mean(),
            'pr_auc': average_precision_score(g[config.TARGET_COL], g['proba']),
            'top20pct_contact_rate': (g['proba'] >= np.quantile(test_aug['proba'], 0.8)).mean(),
        })
    return pd.DataFrame(rows)

by_age = group_perf(test_aug, 'age_band')
by_edu = group_perf(test_aug, 'education')
by_age.to_csv(config.TABLES_DIR / 'fairness_by_age.csv', index=False)
by_edu.to_csv(config.TABLES_DIR / 'fairness_by_education.csv', index=False)
by_age, by_edu


## 9.6 — Three-bullet summary for the compliance slide

- The features the model relies on most.
- Any group disparities and the proposed mitigation (or why none is needed).
- The fallback decision rule (shallow tree) for human-only review cases.
